# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [56]:
! pip install -q schedule pytest 


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [57]:
! pip install -q requests beautifulsoup4


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup
# подключил регулярные выражения чтобы достать количество доступеых книг
import re

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [ ]:
# Возможно у меня проблемы с интернетом и страницы не с первого раза открываются
def get_response(url: str) -> requests.Response:
    """Получает страницу, делая до трёх попыток при сетевом сбое."""
    for attempt in range(1, 4):
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            return response

        except (requests.ConnectionError, requests.Timeout):
            if attempt == 3:
                raise

            delay = 2 * attempt
            print(f"Сбой запроса. Повтор через {delay} сек.: {url}")
            time.sleep(delay)


def get_book_data(book_url: str) -> dict:
   """
   Сборка данных о книге с сайта.

   Args:
      book_url (str): URL книги.

   Returns:
      dict: название, цена, рейтинг, количество, статус наличия,
         описание, другие характеристики. 
   """

   # НАЧАЛО ВАШЕГО РЕШЕНИЯ
   # получим доступ к странице
   response = get_response(book_url)
   # разбираем html
   soup = BeautifulSoup(response.content, "html.parser")

   # название находится в теге h1
   title = soup.find("h1").get_text()
   # вытащим цену из тега p с классом price_color
   price = soup.find("p", class_="price_color").get_text()

   # получаем рейтинг в виде слова из второго класса элемента с тегом p и классом star-rating 
   rating = soup.find("p", class_="star-rating")["class"][1]

   # наличие в классе availability
   availability = soup.find("p", class_="availability").get_text().strip()
   stock = re.search(r"\d+", availability).group()

   # находим блок с заголовком описания по его id
   description_header = soup.find("div", id="product_description")
   # Если описание отсутствует, оставляем пустую строку
   description = ""

   # оказывается у некоторых книжек нету описания!
   if description_header is not None:
      # на одном уровне с этим заголовком находится тег p, внутри которого есть описание
      description_tag = description_header.find_next_sibling("p")
      description = description_tag.get_text(strip=True)

   # остается распарсить таблицу
   table = soup.find("table", class_="table-striped")
   product_info = {}
   for row in table.find_all("tr"):
      product_info[row.find("th").get_text()] = row.find("td").get_text()

   # сборка данных книги в словарь
   return {
      "title": title,
      "price": price,
      "rating": rating,
      "stock": stock,
      "availability": availability,
      "description": description,
      "product_info": product_info
   }
   # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [60]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': '£51.77',
 'rating': 'Three',
 'stock': '22',
 'availability': 'In stock (22 available)',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a tr

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
import json
from urllib.parse import urljoin


def scrape_books(is_save: bool = False) -> list:
    """Сборка всех книг из каталога.

    Args:
        is_save (bool): Если True, сохраняет результат в books_data.txt
            в текущей рабочей папке. По умолчанию False.

    Returns:
        list: Список словарей с данными о книгах.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    books = []
    catalog_url = "https://books.toscrape.com/catalogue/page-1.html"

    while True:
        # получим текущую страницу каталога.
        response = get_response(catalog_url)
        # снова начинаем разбирать html
        soup = BeautifulSoup(response.content, "html.parser")

        # все элементы в каталоге с тегом h3 имеют подэлемент с тегом a, где указана ссылка на книжку
        for heading in soup.find_all("h3"):
            link_tag = heading.find("a")
            # получаем полный путь книги
            book_url = urljoin(catalog_url, link_tag["href"])
            book_data = get_book_data(book_url)
            books.append(book_data)

        # сделаем логирование количества собранных книг на данной странице
        print(f"Собрано книг: {len(books)}")

        # ищем ссылку на следующую страницу.
        next_page = soup.find("li", class_="next")

        # если следующей страницы нет, завершаем цикл
        if next_page is None:
            break

        next_link = next_page.find("a")["href"]
        catalog_url = urljoin(catalog_url, next_link)

    # При необходимости сохраняем весь результат
    if is_save:
        with open("books_data.txt", "w", encoding="utf-8") as file:
            json.dump(books, file, ensure_ascii=False, indent=4)

    return books
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [62]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Сбой запроса. Повтор через 2 сек.: https://books.toscrape.com/catalogue/starving-hearts-triangular-trade-trilogy-1_990/index.html
Собрано книг: 20
Собрано книг: 40
Собрано книг: 60
Собрано книг: 80
Сбой запроса. Повтор через 2 сек.: https://books.toscrape.com/catalogue/lumberjanes-vol-1-beware-the-kitten-holy-lumberjanes-1-4_906/index.html
Собрано книг: 100
Собрано книг: 120
Собрано книг: 140
Собрано книг: 160
Сбой запроса. Повтор через 2 сек.: https://books.toscrape.com/catalogue/so-youve-been-publicly-shamed_832/index.html
Собрано книг: 180
Собрано книг: 200
Сбой запроса. Повтор через 2 сек.: https://books.toscrape.com/catalogue/batman-the-long-halloween-batman_793/index.html
Сбой запроса. Повтор через 2 сек.: https://books.toscrape.com/catalogue/amid-the-chaos_788/index.html
Собрано книг: 220
Собрано книг: 240
Собрано книг: 260
Сбой запроса. Повтор через 2 сек.: https://books.toscrape.com/catalogue/no-dream-is-too-high-life-lessons-from-a-man-who-walked-on-the-moon_722/index.html
Со

## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [6]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
# очистим расписание
schedule.clear()


job = schedule.every().day.at("19:00").do(scrape_books, is_save=True)

# посмотрим, когда ближайший запуск
print("Следующий запуск: ", job.next_run)

while True:
    # проверка расписания
    schedule.run_pending()
    # отдых процесса
    time.sleep(1)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

Следующий запуск:  2026-11-15 19:00:00


KeyboardInterrupt: 

## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [9]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest ../tests/test_scraper.py -v

============================= test session starts =============================
platform win32 -- Python 3.11.9, pytest-9.2.0, pluggy-1.7.0 -- C:\Users\Александр\AppData\Local\Programs\Python\Python311\python.exe
cachedir: .pytest_cache
rootdir: d:\edu\edumipt\python\hw\hw3
configfile: pytest.ini
collecting ... collected 4 items

..\tests\test_scraper.py::test_get_book_data PASSED                      [ 25%]
..\tests\test_scraper.py::test_get_book_data_without_description PASSED  [ 50%]
..\tests\test_scraper.py::test_scrape_books[False] PASSED                [ 75%]
..\tests\test_scraper.py::test_scrape_books[True] PASSED                 [100%]

============================== 4 passed in 0.15s ==============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Нужно было написать парсер страниц книг и каталогов, сделать расписание для этой задачи, написать пару автотестов, а также создать репозиторий. Получился сборщик, который успешно обработал 1000 страниц книг.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Застрял на втором задании, потому что иногда у меня программа не могла получить доступ к странице и поэтому пришлось написать мини-функцию, которая пытается получить ответ от сайта несколько раз, а потом случилось так, что у некоторых книг нет описания и такие случаи не на первых страницах каталога, а так как программа работает долго, это было тяжело выяснить сразу.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Не использовал.`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: По этому заданию вроде бы понятно.`

**5. Самооценка по критериям задания**:

`Ответ: Я считаю, что задание выполнено корректно.`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: ...`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

